In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

In [3]:
df=pd.read_parquet('../data/featured/olist_delivery_features.parquet')

In [4]:
df.head()

,late_delivery,purchase_hour,purchase_month,expected_delivery_days,approval_hours,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
0,0,10,10,15,0.178333,1,0,0,0,0,0
1,0,20,7,19,30.713889,0,0,0,0,1,0
2,0,8,8,26,0.276111,0,0,0,0,0,1
3,0,19,11,26,0.298056,0,1,0,0,0,0
4,0,21,2,12,1.030556,0,0,0,0,1,0


# Training a Baseline Model

## Logistic Regression
- Logistic Regression
    - Simple
    - Fast
    - Probabilities
    - Good baseline
Establish a benchmark

In [5]:
X=df.drop('late_delivery',axis=1)
Y=df.late_delivery

In [6]:
X.head()

,purchase_hour,purchase_month,expected_delivery_days,approval_hours,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
0,10,10,15,0.178333,1,0,0,0,0,0
1,20,7,19,30.713889,0,0,0,0,1,0
2,8,8,26,0.276111,0,0,0,0,0,1
3,19,11,26,0.298056,0,1,0,0,0,0
4,21,2,12,1.030556,0,0,0,0,1,0


In [7]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96461 entries, 0 to 96460
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   purchase_hour           96461 non-null  int32  
 1   purchase_month          96461 non-null  int32  
 2   expected_delivery_days  96461 non-null  int64  
 3   approval_hours          96461 non-null  float64
 4   Monday                  96461 non-null  int64  
 5   Saturday                96461 non-null  int64  
 6   Sunday                  96461 non-null  int64  
 7   Thursday                96461 non-null  int64  
 8   Tuesday                 96461 non-null  int64  
 9   Wednesday               96461 non-null  int64  
dtypes: float64(1), int32(2), int64(7)
memory usage: 6.6 MB


In [8]:
Y.head()

0    0
1    0
2    0
3    0
4    0
Name: late_delivery, dtype: int64

In [9]:
x_train,x_test,y_train,y_test= train_test_split(X,Y,test_size=.20,random_state=20,stratify=Y)

In [10]:
num_cols=['purchase_hour','purchase_month','expected_delivery_days','approval_hours']

In [11]:
scaler=StandardScaler()
x_train[num_cols]=scaler.fit_transform(x_train[num_cols])
x_test[num_cols]=scaler.fit_transform(x_test[num_cols])

In [12]:
x_train.head()

,purchase_hour,purchase_month,expected_delivery_days,approval_hours,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
42860,-0.142572,-0.629950,3.269043,2.522973,0,1,0,0,0,0
32691,0.419117,0.299397,-0.611879,-0.483274,0,0,1,0,0,0
50506,0.231887,1.228745,-0.269444,1.861106,1,0,0,0,0,0
15676,0.793576,0.918963,-0.840168,-0.487962,1,0,0,0,0,0
62062,-1.453179,0.609180,1.100292,-0.489791,0,0,0,0,0,1


In [13]:
lr=LogisticRegression(random_state=20,max_iter=1000)
lr.fit(x_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,20
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [14]:
y_pred=lr.predict(x_test) # predict() , threshold >=0.5 -> 1
y_prob=lr.predict_proba(x_test)[:,1]

In [15]:
y_prob

array([0.11202341, 0.04726428, 0.07435699, ..., 0.08554969, 0.05382263,
       0.0573057 ], shape=(19293,))

In [16]:
print("Accuracy :", accuracy_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("Recall   :", recall_score(y_test,y_pred))
print("F1 Score :", f1_score(y_test,y_pred))
print("ROC AUC  :", roc_auc_score(y_test,y_prob))

Accuracy : 0.9189343285129321
Precision: 1.0
Recall   : 0.0006389776357827476
F1 Score : 0.001277139208173691
ROC AUC  : 0.5847508967601297


In [17]:
cm = confusion_matrix(y_test,y_pred)
print(cm)

[[17728     0]
 [ 1564     1]]


In [18]:
pd.Series(lr.predict_proba(x_test)[:,-1]).describe()

count    19293.000000
mean         0.081108
std          0.021339
min          0.002666
25%          0.066929
50%          0.079670
75%          0.093903
max          0.721018
dtype: float64

In [19]:
y2_prob=lr.predict_proba(x_test)[:,1]

In [20]:
y2_pred=(y2_prob>=.2).astype(int)

In [21]:
print("Accuracy :", accuracy_score(y_test,y2_pred))
print("Precision:", precision_score(y_test,y2_pred))
print("Recall   :", recall_score(y_test,y2_pred))
print("F1 Score :", f1_score(y_test,y2_pred))
print("ROC AUC  :", roc_auc_score(y_test,y2_prob))

Accuracy : 0.9186751671590733
Precision: 0.25
Recall   : 0.0012779552715654952
F1 Score : 0.0025429116338207248
ROC AUC  : 0.5847508967601297


No significant improvements even after decreasing threshold.
- Weakness - Class Imbalance

In [22]:
lr_model2=LogisticRegression(random_state=20,max_iter=100,class_weight='balanced')
lr_model2.fit(x_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,20
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [23]:
y3_pred=lr_model2.predict(x_test)

In [24]:
y3_prob=lr_model2.predict_proba(x_test)[:,1]

In [25]:
print("Accuracy :", accuracy_score(y_test,y3_pred))
print("Precision:", precision_score(y_test,y3_pred))
print("Recall   :", recall_score(y_test,y3_pred))
print("F1 Score :", f1_score(y_test,y3_pred))
print("ROC AUC  :", roc_auc_score(y_test,y3_prob))

Accuracy : 0.5439796817498574
Precision: 0.10060739922694643
Recall   : 0.582108626198083
F1 Score : 0.17156308851224106
ROC AUC  : 0.5838047571538967


In [26]:
cm_model2=confusion_matrix(y_test,y3_pred)
cm_model2

array([[9584, 8144],
       [ 654,  911]])

## Decision Tree

The relationship may be non-linear and rule-based.

In [3]:
df=pd.read_parquet("../data/featured/olist_delivery_features.parquet")

In [4]:
df.head()

,late_delivery,purchase_hour,purchase_month,expected_delivery_days,approval_hours,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
0,0,10,10,15,0.178333,1,0,0,0,0,0
1,0,20,7,19,30.713889,0,0,0,0,1,0
2,0,8,8,26,0.276111,0,0,0,0,0,1
3,0,19,11,26,0.298056,0,1,0,0,0,0
4,0,21,2,12,1.030556,0,0,0,0,1,0


In [5]:
X=df.drop('late_delivery',axis=1)
Y=df.late_delivery

In [6]:
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=.20,random_state=20)

In [7]:
dct_model1=DecisionTreeClassifier(random_state=20) # internal randomness of the algorithm itself during

In [8]:
dct_model1.fit(x_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,20
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [9]:
y_pred=dct_model1.predict(x_test)
y_prob=dct_model1.predict_proba(x_test)[:,1]

In [10]:
print("Accuracy: ",accuracy_score(y_test,y_pred))
print("Precision: ",precision_score(y_test,y_pred))
print("Recall: ",recall_score(y_test,y_pred))
print("F1_Score: ",f1_score(y_test,y_pred))
print("ROC_AUC_Score: ",roc_auc_score(y_test,y_prob))

Accuracy:  0.8559581195252164
Precision:  0.13921113689095127
Recall:  0.1563517915309446
F1_Score:  0.1472844430806996
ROC_AUC_Score:  0.5360118465669278


In [11]:
conf_mat=confusion_matrix(y_test,y_pred)
conf_mat

array([[16274,  1484],
       [ 1295,   240]])

In [12]:
print(f"Train Score: {dct_model1.score(x_train,y_train)}")
print(f"Test Score:  {dct_model1.score(x_test,y_test)}")


Train Score: 0.9996371552975326
Test Score:  0.8559581195252164


HyperParameter Tuning

In [13]:
param_grid = {
    'class_weight': ['balanced', None],
    'max_depth': [3, 5, 10, None],
    'min_samples_split':[2,5,10,20],
    'min_samples_leaf':[1,2,5,10],
    'criterion': ['gini', 'entropy']
}


In [14]:
grid_search=GridSearchCV(
    estimator=dct_model1, # take my basic Decision Tree model (base_dct) and use it as the blueprint for all the tests.
    param_grid=param_grid,
    cv=5,
    scoring='recall', # focuses strictly on catching late deliveries
    n_jobs=-1
)

In [16]:
grid_search.fit(x_train,y_train)

,estimator,DecisionTreeC...ndom_state=20)
,param_grid,"{'class_weight': ['balanced', None], 'criterion': ['gini', 'entropy'], 'max_depth': [3, 5, ...], 'min_samples_leaf': [1, 2, ...], ...}"
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'entropy'


In [17]:
print("Best Parameters found: ",grid_search.best_params_)

Best Parameters found:  {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}


In [20]:
best_dct_model=grid_search.best_estimator_

In [22]:
y_pred_new=best_dct_model.predict(x_test)
y_prob_new=best_dct_model.predict_proba(x_test)[:,1]

In [23]:
print("Accuracy: ",accuracy_score(y_test,y_pred_new))
print("Precision: ",precision_score(y_test,y_pred_new))
print("Recall: ",recall_score(y_test,y_pred_new))
print("F1_Score: ",f1_score(y_test,y_pred_new))
print("ROC_AUC_Score: ",roc_auc_score(y_test,y_prob_new))

Accuracy:  0.7279842429896853
Precision:  0.16029277218664226
Recall:  0.5706840390879478
F1_Score:  0.2502857142857143
ROC_AUC_Score:  0.6949256618020121


In [24]:
cfm2=confusion_matrix(y_test,y_pred_new)
cfm2

array([[13169,  4589],
       [  659,   876]])

In [25]:
print(f"Train Score: {best_dct_model.score(x_train,y_train)}")
print(f"Test Score:  {best_dct_model.score(x_test,y_test)}")


Train Score: 0.7307173958117354
Test Score:  0.7279842429896853


Feature Importance

In [31]:
feature_imp=pd.DataFrame({
    "Feature":X.columns,
    "Importance":best_dct_model.feature_importances_
})

In [32]:
feature_imp.sort_values(
    by="Importance",
    ascending=False
)

,Feature,Importance
1,purchase_month,0.603108
2,expected_delivery_days,0.340202
3,approval_hours,0.054229
6,Sunday,0.001992
0,purchase_hour,0.000469
4,Monday,0.000000
5,Saturday,0.000000
7,Thursday,0.000000
8,Tuesday,0.000000
9,Wednesday,0.000000


## Random Forest

The tuned Decision Tress improved recall and ROC-AUC but remains a high-variance model because it relies on a single tree.
Random Forest is evaluated to reduce overfitting and improve generalization by combining predictions from multiple decisions trees.

In [6]:
from sklearn.ensemble import RandomForestClassifier

In [4]:
X=df.drop('late_delivery',axis=1)
Y=df.late_delivery

In [5]:
x_train,x_test,y_train,y_test= train_test_split(X,Y,test_size=.20,random_state=20,stratify=Y)

In [7]:
rf_model1=RandomForestClassifier(
    random_state=20
)
rf_model1.fit(x_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
y_pred=rf_model1.predict(x_test)
y_prob=rf_model1.predict_proba(x_test)

In [14]:
print("Accuracy: ",accuracy_score(y_test,y_pred))
print("Precision: ",precision_score(y_test,y_pred))
print("Recall: ",recall_score(y_test,y_pred))
print("F1score: ",f1_score(y_test,y_pred))
print("ROC_AUC: ",roc_auc_score(y_test,y_prob[:,1]))

Accuracy:  0.9117296428756544
Precision:  0.2636986301369863
Recall:  0.049201277955271565
F1score:  0.08292945611200861
ROC_AUC:  0.6408263204864996


In [15]:
print("Train Score: ",rf_model1.score(x_train,y_train))
print("Test Score: ", rf_model1.score(x_test,y_test))

Train Score:  0.9995205266431682
Test Score:  0.9117296428756544
